# 01 - Dataset1 Varroa Feature Engineering

Bu notebook, Dataset1 patch metadata dosyalarından model eğitiminde kullanılacak feature tablolarını üretir.

Ham görüntüler değiştirilmez. Feature tabloları `data/features/dataset1_varroa` altında saklanır.


## 1. Kütüphaneler

Bu bölümde görüntü işleme, özellik çıkarımı, PCA ve parquet çıktıları için kullanılan kütüphaneler yüklenir.


In [ ]:
from pathlib import Path
from datetime import datetime
from collections import OrderedDict
import gc
import math
import warnings

import cv2
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from skimage.feature import hog, local_binary_pattern
from sklearn.decomposition import PCA

warnings.filterwarnings("ignore")

try:
    import pyarrow  # noqa: F401
    PARQUET_ENGINE = "pyarrow"
    print(f"[OK] pyarrow available: {pyarrow.__version__}")
except ImportError as exc:
    raise ImportError(
        "pyarrow is required for parquet feature output. "
        "Install it with: conda install -c conda-forge pyarrow"
    ) from exc


## 2. Ayarlar

Dataset adı, patch setleri, feature setleri ve overwrite davranışı burada tanımlanır.


In [ ]:
DATASET_NAME = "dataset1_varroa"
DATASET_DISPLAY_NAME = "Dataset1 Varroa"
STAGE_NAME = "03_feature_engineering"
NOTEBOOK_NAME = "01_dataset1_varroa_feature_engineering"

RANDOM_STATE = 42

# Yazma davranışı
OVERWRITE_FEATURES = False
OVERWRITE_TABLES = False
OVERWRITE_FIGURES = False

# Feature konfigürasyonu
HOG_ORIENTATIONS = 9
HOG_CELLS_PER_BLOCK = (2, 2)
HOG_BLOCK_NORM = "L2-Hys"
HOG_TRANSFORM_SQRT = True

HSV_H_BINS = 16
HSV_S_BINS = 8
HSV_V_BINS = 8

LBP_RADIUS = 1
LBP_POINTS = 8 * LBP_RADIUS
LBP_METHOD = "uniform"

PCA_EXPLAINED_VARIANCE = 0.95

# HOG32 bu aşamadaki feature setinden çıkarıldı.
HOG_CONFIGS = {
    "hog8": {"pixels_per_cell": (8, 8)},
    "hog16": {"pixels_per_cell": (16, 16)},
}

FEATURE_SETS = [
    {"feature_set": "hog8_hsv_lbp", "hog_variant": "hog8", "use_pca": False, "include_hsv": True, "include_lbp": True},
    {"feature_set": "hog8_hsv", "hog_variant": "hog8", "use_pca": False, "include_hsv": True, "include_lbp": False},
    {"feature_set": "hog8_only", "hog_variant": "hog8", "use_pca": False, "include_hsv": False, "include_lbp": False},
    {"feature_set": "hog16_hsv_lbp", "hog_variant": "hog16", "use_pca": False, "include_hsv": True, "include_lbp": True},
    {"feature_set": "hog16_hsv", "hog_variant": "hog16", "use_pca": False, "include_hsv": True, "include_lbp": False},
    {"feature_set": "hog16_only", "hog_variant": "hog16", "use_pca": False, "include_hsv": False, "include_lbp": False},
    {"feature_set": "hsv_lbp_only", "hog_variant": None, "use_pca": False, "include_hsv": True, "include_lbp": True},
    {"feature_set": "hog8_pca_hsv_lbp", "hog_variant": "hog8", "use_pca": True, "include_hsv": True, "include_lbp": True},
    {"feature_set": "hog8_pca_hsv", "hog_variant": "hog8", "use_pca": True, "include_hsv": True, "include_lbp": False},
    {"feature_set": "hog8_pca_only", "hog_variant": "hog8", "use_pca": True, "include_hsv": False, "include_lbp": False},
    {"feature_set": "hog16_pca_hsv_lbp", "hog_variant": "hog16", "use_pca": True, "include_hsv": True, "include_lbp": True},
    {"feature_set": "hog16_pca_hsv", "hog_variant": "hog16", "use_pca": True, "include_hsv": True, "include_lbp": False},
    {"feature_set": "hog16_pca_only", "hog_variant": "hog16", "use_pca": True, "include_hsv": False, "include_lbp": False},
]

PATCH_SETS = ['centered80_balanced', 'centered80_neg4x', 'centered96_balanced', 'centered96_neg4x', 'centered112_balanced', 'centered112_neg4x']

DISPLAY_PREVIEW_ROWS = 5
IMAGE_CACHE_MAX_ITEMS = 64
REPORT_MAX_TABLE_ROWS = 50

NOTEBOOK_STARTED_AT = datetime.now()
print(f"[START] {NOTEBOOK_NAME} at {NOTEBOOK_STARTED_AT:%Y-%m-%d %H:%M:%S}")
print(f"[INFO] Dataset: {DATASET_NAME}")
print(f"[INFO] Patch sets: {len(PATCH_SETS)}")
print(f"[INFO] Feature sets: {len(FEATURE_SETS)}")


## 3. Yardımcı Fonksiyonlar ve Dosya Yolları

Proje kökü bulunur; metadata, feature, tablo ve görsel çıktı klasörleri hazırlanır.


In [ ]:
def find_project_root(start_path=None):
    if start_path is None:
        try:
            start_path = Path(__file__).resolve()
        except NameError:
            start_path = Path.cwd().resolve()

    start_path = Path(start_path).resolve()
    candidates = [start_path] + list(start_path.parents)
    for candidate in candidates:
        if (candidate / "data").exists() and (candidate / "approaches").exists():
            return candidate

    raise FileNotFoundError(
        "Proje kökü bulunamadı. Notebook'u proje klasörü içinde çalıştırdığından emin ol."
    )


PROJECT_ROOT = find_project_root()
APPROACH_NAME = "traditional_ml"
APPROACH_DIR = PROJECT_ROOT / "approaches" / APPROACH_NAME
NOTEBOOK_DIR = APPROACH_DIR / "notebooks" / STAGE_NAME
OUTPUT_DIR = APPROACH_DIR / "outputs" / STAGE_NAME / NOTEBOOK_NAME
TABLES_DIR = OUTPUT_DIR / "tables"
FIGURES_DIR = OUTPUT_DIR / "figures"

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw" / DATASET_NAME
METADATA_DIR = PROJECT_ROOT / "data" / "metadata" / DATASET_NAME
FEATURES_DIR = PROJECT_ROOT / "data" / "features" / DATASET_NAME
PCA_ARTIFACT_DIR = FEATURES_DIR / "pca_artifacts"
FEATURE_MANIFEST_PATH = FEATURES_DIR / "feature_manifest.csv"

for directory in [FEATURES_DIR, PCA_ARTIFACT_DIR, TABLES_DIR, FIGURES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

required_dirs = {
    "PROJECT_ROOT": PROJECT_ROOT,
    "RAW_DATA_DIR": RAW_DATA_DIR,
    "METADATA_DIR": METADATA_DIR,
}

for name, path in required_dirs.items():
    if not path.exists():
        raise FileNotFoundError(f"{name} bulunamadı: {path}")

print(f"[OK] Proje kökü: {PROJECT_ROOT}")
print(f"[OK] Ham veri: {RAW_DATA_DIR}")
print(f"[OK] Metadata: {METADATA_DIR}")
print(f"[OK] Feature çıktısı: {FEATURES_DIR}")
print(f"[OK] Notebook çıktısı: {OUTPUT_DIR}")

## 4. Kayıt ve çıktı Yardımcıları

Tablo, parquet, joblib ve görsel kayıtları için ortak yardımcı fonksiyonlar tanımlanır.


In [ ]:
def log_output(message):
    print(message)


def log_section(title):
    print()
    print("=" * 80)
    print(title)
    print("=" * 80)


def log_dataframe(title, df, max_rows=REPORT_MAX_TABLE_ROWS):
    print()
    print(f"[DATAFRAME] {title}")
    if df is None:
        print("DataFrame yok.")
        return
    print(f"Shape: {df.shape}")
    display(df.head(max_rows))


def log_figure(title, description, figure_path):
    figure_path = Path(figure_path)
    print(f"[FIGURE] {title}: {figure_path}")


def timestamp_now():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


def relative_path(path):
    path = Path(path)

    try:
        return path.resolve().relative_to(PROJECT_ROOT.resolve()).as_posix()
    except Exception:
        return path.as_posix()


def save_dataframe_csv(df, output_path, overwrite=True, index=False, note=""):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    if output_path.exists() and not overwrite:
        try:
            existing = pd.read_csv(output_path)
            print(f"[LOAD] Existing CSV loaded: {output_path}")
            return existing
        except pd.errors.EmptyDataError:
            print(f"[SKIP] Existing CSV is empty; current DataFrame kept in memory: {output_path}")
            return df

    if output_path.exists() and overwrite:
        print(f"[OVERWRITE] Replacing CSV: {output_path}")

    df.to_csv(output_path, index=index, encoding="utf-8-sig")
    print(f"[SAVE] CSV saved: {output_path}")
    return df


def save_parquet(df, output_path, overwrite=False, note=""):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    if output_path.exists() and not overwrite:
        print(f"[LOAD] Existing parquet loaded: {output_path}")
        existing = pd.read_parquet(output_path)
        return existing, "loaded_existing"

    if output_path.exists() and overwrite:
        print(f"[OVERWRITE] Replacing parquet: {output_path}")

    df.to_parquet(output_path, index=False, engine=PARQUET_ENGINE)
    print(f"[SAVE] Parquet saved: {output_path}")
    return df, "created_or_overwritten"


def save_joblib(obj, output_path, overwrite=False, note=""):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    if output_path.exists() and not overwrite:
        print(f"[SKIP] Existing joblib kept: {output_path}")
        return "skipped_existing"

    if output_path.exists() and overwrite:
        print(f"[OVERWRITE] Replacing joblib: {output_path}")

    joblib.dump(obj, output_path)
    print(f"[SAVE] Joblib saved: {output_path}")
    return "created_or_overwritten"


def save_current_figure(output_path, title, description=""):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    if output_path.exists() and not OVERWRITE_FIGURES:
        plt.close()
        print(f"[SKIP] Existing figure kept: {output_path}")
        return output_path

    if output_path.exists() and OVERWRITE_FIGURES:
        print(f"[OVERWRITE] Replacing figure: {output_path}")

    plt.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.close()
    log_figure(title, description, output_path)
    return output_path


## 5. Feature Set Konfigürasyonu

HOG, HSV, LBP ve PCA tabanlı feature setlerinin özet konfigürasyonu kaydedilir.


In [ ]:
log_section("01 FEATURE ENGINEERING BAŞLADI")

feature_set_config_df = pd.DataFrame(FEATURE_SETS)
feature_set_config_df["dataset_name"] = DATASET_NAME
feature_set_config_df["pca_explained_variance"] = np.where(
    feature_set_config_df["use_pca"],
    PCA_EXPLAINED_VARIANCE,
    np.nan,
)

hog_config_rows = []
for hog_variant, cfg in HOG_CONFIGS.items():
    hog_config_rows.append({
        "hog_variant": hog_variant,
        "pixels_per_cell": str(cfg["pixels_per_cell"]),
        "orientations": HOG_ORIENTATIONS,
        "cells_per_block": str(HOG_CELLS_PER_BLOCK),
        "block_norm": HOG_BLOCK_NORM,
        "transform_sqrt": HOG_TRANSFORM_SQRT,
    })
hog_config_df = pd.DataFrame(hog_config_rows)

hsv_lbp_config_df = pd.DataFrame([{
    "hsv_h_bins": HSV_H_BINS,
    "hsv_s_bins": HSV_S_BINS,
    "hsv_v_bins": HSV_V_BINS,
    "lbp_radius": LBP_RADIUS,
    "lbp_points": LBP_POINTS,
    "lbp_method": LBP_METHOD,
}])

save_dataframe_csv(feature_set_config_df, TABLES_DIR / "feature_set_config.csv", overwrite=OVERWRITE_TABLES)
save_dataframe_csv(hog_config_df, TABLES_DIR / "hog_config.csv", overwrite=OVERWRITE_TABLES)
save_dataframe_csv(hsv_lbp_config_df, TABLES_DIR / "hsv_lbp_config.csv", overwrite=OVERWRITE_TABLES)

log_dataframe("Feature Set Configuration", feature_set_config_df)
log_dataframe("HOG Configuration", hog_config_df)
log_dataframe("HSV / LBP Configuration", hsv_lbp_config_df)


## 6. Metadata Girdi Envanteri

Patch metadata dosyalarının varlığı ve satır sayıları kontrol edilir.


In [ ]:
log_section("02 METADATA INPUT INVENTORY")

metadata_inventory_rows = []

for patch_set in PATCH_SETS:
    metadata_path = METADATA_DIR / f"patch_metadata_{patch_set}.csv"
    if not metadata_path.exists():
        raise FileNotFoundError(f"Patch metadata file not found: {metadata_path}")

    df_preview = pd.read_csv(metadata_path)
    split_counts = df_preview["split"].value_counts().to_dict() if "split" in df_preview.columns else {}

    metadata_inventory_rows.append({
        "dataset_name": DATASET_NAME,
        "patch_set": patch_set,
        "metadata_path": relative_path(metadata_path),
        "exists": metadata_path.exists(),
        "row_count": len(df_preview),
        "column_count": len(df_preview.columns),
        "train_count": split_counts.get("train", 0),
        "valid_count": split_counts.get("valid", 0),
        "test_count": split_counts.get("test", 0),
        "positive_count": int((df_preview["patch_label"] == 1).sum()) if "patch_label" in df_preview.columns else np.nan,
        "negative_count": int((df_preview["patch_label"] == 0).sum()) if "patch_label" in df_preview.columns else np.nan,
    })

metadata_input_inventory_df = pd.DataFrame(metadata_inventory_rows)
save_dataframe_csv(metadata_input_inventory_df, TABLES_DIR / "metadata_input_inventory.csv", overwrite=OVERWRITE_TABLES)
log_dataframe("Metadata Input Inventory", metadata_input_inventory_df)


## 7. Feature Dosya Planı

Her patch set ve feature set için üretilecek parquet dosyaları planlanır.


In [ ]:
log_section("03 FEATURE FILE PLAN")

feature_file_plan_rows = []

for patch_set in PATCH_SETS:
    metadata_path = METADATA_DIR / f"patch_metadata_{patch_set}.csv"
    patch_size = int(patch_set.replace("centered", "").split("_")[0])
    ratio_variant = patch_set.split("_")[-1]

    for feature_cfg in FEATURE_SETS:
        feature_set = feature_cfg["feature_set"]
        output_path = FEATURES_DIR / f"{patch_set}__{feature_set}.parquet"

        feature_file_plan_rows.append({
            "dataset_name": DATASET_NAME,
            "patch_set": patch_set,
            "patch_size": patch_size,
            "ratio_variant": ratio_variant,
            "feature_set": feature_set,
            "hog_variant": feature_cfg["hog_variant"],
            "use_pca": feature_cfg["use_pca"],
            "include_hsv": feature_cfg["include_hsv"],
            "include_lbp": feature_cfg["include_lbp"],
            "metadata_path": relative_path(metadata_path),
            "feature_file_path": relative_path(output_path),
            "planned_status": "PLANNED",
        })

feature_file_plan_df = pd.DataFrame(feature_file_plan_rows)
save_dataframe_csv(feature_file_plan_df, TABLES_DIR / "feature_file_plan.csv", overwrite=OVERWRITE_TABLES)
log_dataframe("Feature File Plan", feature_file_plan_df)


## 8. Patch Crop ve Feature Yardımcıları

Patch crop rekonstrüksiyonu ve HOG, HSV, LBP özellik çıkarımı için fonksiyonlar hazırlanır.


In [ ]:
log_section("04 PATCH CROP AND FEATURE EXTRACTION HELPERS")


def resolve_project_path(path_value):
    """Resolve absolute or project-relative paths stored in metadata."""
    if pd.isna(path_value):
        return None

    path = Path(str(path_value))
    if path.is_absolute():
        return path

    candidate = PROJECT_ROOT / path
    if candidate.exists():
        return candidate

    # Stage 02 usually stores source image paths. This fallback helps if paths are relative to data/raw.
    candidate = RAW_DATA_DIR / path
    if candidate.exists():
        return candidate

    return path


image_cache = OrderedDict()


def read_image_cached(image_path):
    image_path = Path(image_path)

    if image_path in image_cache:
        image_cache.move_to_end(image_path)
        return image_cache[image_path]

    image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if image is None:
        raise FileNotFoundError(f"Image could not be read: {image_path}")

    image_cache[image_path] = image
    if len(image_cache) > IMAGE_CACHE_MAX_ITEMS:
        image_cache.popitem(last=False)

    return image


def crop_patch_from_row(row):
    image_path = resolve_project_path(row["source_image_path"])
    image = read_image_cached(image_path)

    x1 = int(row["x1"])
    y1 = int(row["y1"])
    x2 = int(row["x2"])
    y2 = int(row["y2"])

    patch = image[y1:y2, x1:x2]

    expected_size = int(row["patch_size"])
    if patch.size == 0:
        raise ValueError(f"Empty crop for patch_id={row.get('patch_id')} from image={image_path}")

    if patch.shape[0] != expected_size or patch.shape[1] != expected_size:
        raise ValueError(
            f"Unexpected crop shape for patch_id={row.get('patch_id')}: "
            f"expected {expected_size}x{expected_size}, got {patch.shape[:2]}"
        )

    return patch


def patch_to_gray_float(patch_bgr):
    gray = cv2.cvtColor(patch_bgr, cv2.COLOR_BGR2GRAY)
    return gray.astype(np.float32) / 255.0


def extract_hog_features(patch_bgr, hog_variant):
    gray_float = patch_to_gray_float(patch_bgr)
    cfg = HOG_CONFIGS[hog_variant]
    return hog(
        gray_float,
        orientations=HOG_ORIENTATIONS,
        pixels_per_cell=cfg["pixels_per_cell"],
        cells_per_block=HOG_CELLS_PER_BLOCK,
        block_norm=HOG_BLOCK_NORM,
        transform_sqrt=HOG_TRANSFORM_SQRT,
        feature_vector=True,
    ).astype(np.float32)


def extract_hsv_histogram(patch_bgr):
    hsv = cv2.cvtColor(patch_bgr, cv2.COLOR_BGR2HSV)

    h_hist = cv2.calcHist([hsv], [0], None, [HSV_H_BINS], [0, 180]).flatten()
    s_hist = cv2.calcHist([hsv], [1], None, [HSV_S_BINS], [0, 256]).flatten()
    v_hist = cv2.calcHist([hsv], [2], None, [HSV_V_BINS], [0, 256]).flatten()

    def normalize(hist):
        total = hist.sum()
        if total <= 0:
            return np.zeros_like(hist, dtype=np.float32)
        return (hist / total).astype(np.float32)

    return np.concatenate([normalize(h_hist), normalize(s_hist), normalize(v_hist)]).astype(np.float32)


def extract_lbp_histogram(patch_bgr):
    gray = cv2.cvtColor(patch_bgr, cv2.COLOR_BGR2GRAY)
    lbp = local_binary_pattern(gray, P=LBP_POINTS, R=LBP_RADIUS, method=LBP_METHOD)

    # Uniform LBP returns values from 0 to P + 1.
    bins = np.arange(0, LBP_POINTS + 3)
    hist, _ = np.histogram(lbp.ravel(), bins=bins, range=(0, LBP_POINTS + 2))

    total = hist.sum()
    if total <= 0:
        return np.zeros(LBP_POINTS + 2, dtype=np.float32)

    return (hist / total).astype(np.float32)


def make_hog_columns(hog_variant, n_features):
    return [f"{hog_variant}_{idx:04d}" for idx in range(n_features)]


def make_hog_pca_columns(hog_variant, n_features):
    return [f"{hog_variant}_pca_{idx:04d}" for idx in range(n_features)]


def make_hsv_columns():
    return (
        [f"hsv_h_{idx:02d}" for idx in range(HSV_H_BINS)]
        + [f"hsv_s_{idx:02d}" for idx in range(HSV_S_BINS)]
        + [f"hsv_v_{idx:02d}" for idx in range(HSV_V_BINS)]
    )


def make_lbp_columns():
    return [f"lbp_{idx:02d}" for idx in range(LBP_POINTS + 2)]


METADATA_COLUMNS_TO_KEEP = [
    "patch_id",
    "dataset_name",
    "split",
    "patch_set",
    "patch_size",
    "ratio_variant",
    "patch_label",
    "patch_type",
    "source_image_path",
    "x1",
    "y1",
    "x2",
    "y2",
    "crop_width",
    "crop_height",
    "image_width",
    "image_height",
    "negative_type",
    "requested_negative_zone",
    "actual_negative_zone",
    "actual_normalized_center_distance",
    "was_shifted_to_fit_image",
    "source_bbox_class_id",
    "source_bbox_class_name",
    "source_bbox_width_px",
    "source_bbox_height_px",
    "source_bbox_area_px",
    "flags",
]


def prepare_metadata_subset(metadata_df):
    subset = metadata_df.copy()
    for col in METADATA_COLUMNS_TO_KEEP:
        if col not in subset.columns:
            subset[col] = pd.NA
    return subset[METADATA_COLUMNS_TO_KEEP].copy()


def dataframe_from_matrix(matrix, columns):
    return pd.DataFrame(matrix, columns=columns)


def validate_no_nan_inf(feature_df, feature_columns):
    if not feature_columns:
        return {"nan_count": 0, "inf_count": 0}

    values = feature_df[feature_columns].to_numpy(dtype=np.float32, copy=False)
    return {
        "nan_count": int(np.isnan(values).sum()),
        "inf_count": int(np.isinf(values).sum()),
    }


print("[OK] Feature extraction helper functions are ready.")


## 9. Feature Boyut Önizlemesi

HOG, HSV, LBP ve PCA varyantları için beklenen feature boyutları özetlenir.


In [ ]:
log_section("05 HOG / HSV / LBP FEATURE DIMENSION PREVIEW")

dimension_rows = []

for patch_set in PATCH_SETS:
    patch_size = int(patch_set.replace("centered", "").split("_")[0])
    dummy_patch = np.zeros((patch_size, patch_size, 3), dtype=np.uint8)

    for hog_variant in HOG_CONFIGS:
        hog_vector = extract_hog_features(dummy_patch, hog_variant)
        dimension_rows.append({
            "dataset_name": DATASET_NAME,
            "patch_set": patch_set,
            "patch_size": patch_size,
            "feature_component": hog_variant,
            "feature_dimension": len(hog_vector),
        })

    hsv_vector = extract_hsv_histogram(dummy_patch)
    lbp_vector = extract_lbp_histogram(dummy_patch)

    dimension_rows.append({
        "dataset_name": DATASET_NAME,
        "patch_set": patch_set,
        "patch_size": patch_size,
        "feature_component": "hsv",
        "feature_dimension": len(hsv_vector),
    })
    dimension_rows.append({
        "dataset_name": DATASET_NAME,
        "patch_set": patch_set,
        "patch_size": patch_size,
        "feature_component": "lbp",
        "feature_dimension": len(lbp_vector),
    })

feature_dimension_preview_df = pd.DataFrame(dimension_rows)
save_dataframe_csv(feature_dimension_preview_df, TABLES_DIR / "feature_dimension_preview.csv", overwrite=OVERWRITE_TABLES)
log_dataframe("Feature Dimension Preview", feature_dimension_preview_df)


## 10. PCA Yardımcı Fonksiyonları

PCA artifactlarının train split üzerinden oluşturulması veya mevcutsa yüklenmesi sağlanır.


In [ ]:
log_section("06 PCA HELPER FUNCTIONS")


def fit_or_load_pca(hog_matrix, split_values, patch_set, hog_variant):
    """Fit PCA on train split only, then transform all splits with the same PCA object."""
    pca_path = PCA_ARTIFACT_DIR / f"{patch_set}__{hog_variant}_pca.joblib"
    train_mask = np.asarray(split_values) == "train"

    if train_mask.sum() == 0:
        raise ValueError(f"No train rows found for PCA fitting: {patch_set}, {hog_variant}")

    if pca_path.exists() and not OVERWRITE_FEATURES:
        payload = joblib.load(pca_path)
        pca = payload["pca"]
        action = "loaded_existing"
        print(f"[LOAD] Existing PCA artifact loaded: {pca_path}")
    else:
        pca = PCA(n_components=PCA_EXPLAINED_VARIANCE, svd_solver="full")
        pca.fit(hog_matrix[train_mask])
        payload = {
            "pca": pca,
            "dataset_name": DATASET_NAME,
            "patch_set": patch_set,
            "hog_variant": hog_variant,
            "pca_explained_variance_target": PCA_EXPLAINED_VARIANCE,
            "fit_split": "train",
            "created_at": timestamp_now(),
        }
        action = save_joblib(payload, pca_path, overwrite=OVERWRITE_FEATURES, note=f"{patch_set} {hog_variant} PCA")

    transformed = pca.transform(hog_matrix).astype(np.float32)
    summary = {
        "dataset_name": DATASET_NAME,
        "patch_set": patch_set,
        "hog_variant": hog_variant,
        "pca_artifact_path": relative_path(pca_path),
        "pca_action": action,
        "pca_fit_split": "train",
        "train_row_count": int(train_mask.sum()),
        "original_hog_dimension": int(hog_matrix.shape[1]),
        "pca_n_components_actual": int(pca.n_components_),
        "pca_explained_variance_target": PCA_EXPLAINED_VARIANCE,
        "pca_explained_variance_actual": float(np.sum(pca.explained_variance_ratio_)),
    }
    return transformed, summary


print("[OK] PCA helper functions are ready.")


## 11. Feature Dosyalarının Üretimi

Patch metadata dosyalarından parquet feature tabloları ve feature manifest dosyası üretilir ya da mevcut dosyalar okunur.


In [ ]:
log_section("07 FEATURE FILE GENERATION")

feature_manifest_rows = []
feature_generation_status_rows = []
pca_artifact_summary_rows = []
crop_error_rows = []
nan_inf_rows = []

for patch_index, patch_set in enumerate(PATCH_SETS, start=1):
    log_section(f"07.{patch_index:02d} Processing patch set: {patch_set}")

    metadata_path = METADATA_DIR / f"patch_metadata_{patch_set}.csv"
    metadata_df = pd.read_csv(metadata_path)
    metadata_subset_df = prepare_metadata_subset(metadata_df)

    patch_size = int(metadata_df["patch_size"].iloc[0])
    ratio_variant = str(metadata_df["ratio_variant"].iloc[0])
    split_values = metadata_df["split"].astype(str).to_numpy()

    log_output(f"[RUN] Extracting base features for {patch_set}. Rows: {len(metadata_df)}")

    hog_matrices = {hog_variant: [] for hog_variant in HOG_CONFIGS}
    hsv_features = []
    lbp_features = []

    for row_index, row in metadata_df.iterrows():
        try:
            patch = crop_patch_from_row(row)

            for hog_variant in HOG_CONFIGS:
                hog_matrices[hog_variant].append(extract_hog_features(patch, hog_variant))

            hsv_features.append(extract_hsv_histogram(patch))
            lbp_features.append(extract_lbp_histogram(patch))

        except Exception as exc:
            crop_error_rows.append({
                "dataset_name": DATASET_NAME,
                "patch_set": patch_set,
                "row_index": row_index,
                "patch_id": row.get("patch_id", None),
                "source_image_path": row.get("source_image_path", None),
                "error": repr(exc),
            })
            raise

    for hog_variant in HOG_CONFIGS:
        hog_matrices[hog_variant] = np.vstack(hog_matrices[hog_variant]).astype(np.float32)

    hsv_matrix = np.vstack(hsv_features).astype(np.float32)
    lbp_matrix = np.vstack(lbp_features).astype(np.float32)

    pca_matrices = {}
    pca_summaries = {}

    for hog_variant in HOG_CONFIGS:
        transformed, pca_summary = fit_or_load_pca(
            hog_matrices[hog_variant],
            split_values,
            patch_set,
            hog_variant,
        )
        pca_matrices[hog_variant] = transformed
        pca_summaries[hog_variant] = pca_summary
        pca_artifact_summary_rows.append(pca_summary)

    for feature_cfg in FEATURE_SETS:
        feature_set = feature_cfg["feature_set"]
        output_path = FEATURES_DIR / f"{patch_set}__{feature_set}.parquet"

        if output_path.exists() and not OVERWRITE_FEATURES:
            print(f"[SKIP] Existing feature file found: {output_path}")
            existing_df = pd.read_parquet(output_path)
            action = "loaded_existing"
            feature_columns = [col for col in existing_df.columns if col not in METADATA_COLUMNS_TO_KEEP]
            split_counts = existing_df["split"].value_counts().to_dict()
            label_counts = existing_df["patch_label"].value_counts().to_dict()
            nan_inf = validate_no_nan_inf(existing_df, feature_columns)
        else:
            component_dfs = []
            feature_columns = []

            hog_variant = feature_cfg["hog_variant"]

            if hog_variant is not None:
                if feature_cfg["use_pca"]:
                    hog_component = pca_matrices[hog_variant]
                    hog_columns = make_hog_pca_columns(hog_variant, hog_component.shape[1])
                else:
                    hog_component = hog_matrices[hog_variant]
                    hog_columns = make_hog_columns(hog_variant, hog_component.shape[1])

                component_dfs.append(dataframe_from_matrix(hog_component, hog_columns))
                feature_columns.extend(hog_columns)

            if feature_cfg["include_hsv"]:
                hsv_columns = make_hsv_columns()
                component_dfs.append(dataframe_from_matrix(hsv_matrix, hsv_columns))
                feature_columns.extend(hsv_columns)

            if feature_cfg["include_lbp"]:
                lbp_columns = make_lbp_columns()
                component_dfs.append(dataframe_from_matrix(lbp_matrix, lbp_columns))
                feature_columns.extend(lbp_columns)

            feature_df = pd.concat([metadata_subset_df.reset_index(drop=True)] + component_dfs, axis=1)
            nan_inf = validate_no_nan_inf(feature_df, feature_columns)

            feature_df, action = save_parquet(
                feature_df,
                output_path,
                overwrite=OVERWRITE_FEATURES,
                note=f"{patch_set} {feature_set}",
            )

            split_counts = feature_df["split"].value_counts().to_dict()
            label_counts = feature_df["patch_label"].value_counts().to_dict()

        pca_summary = pca_summaries.get(feature_cfg["hog_variant"], {}) if feature_cfg["use_pca"] else {}

        row_summary = {
            "dataset_name": DATASET_NAME,
            "patch_set": patch_set,
            "patch_size": patch_size,
            "ratio_variant": ratio_variant,
            "feature_set": feature_set,
            "hog_variant": feature_cfg["hog_variant"],
            "is_pca": bool(feature_cfg["use_pca"]),
            "include_hsv": bool(feature_cfg["include_hsv"]),
            "include_lbp": bool(feature_cfg["include_lbp"]),
            "feature_file_path": relative_path(output_path),
            "file_format": "parquet",
            "row_count": int(len(metadata_df)),
            "metadata_column_count": int(len(METADATA_COLUMNS_TO_KEEP)),
            "feature_column_count": int(len(feature_columns)),
            "total_column_count": int(len(METADATA_COLUMNS_TO_KEEP) + len(feature_columns)),
            "train_row_count": int(split_counts.get("train", 0)),
            "valid_row_count": int(split_counts.get("valid", 0)),
            "test_row_count": int(split_counts.get("test", 0)),
            "positive_count": int(label_counts.get(1, 0)),
            "negative_count": int(label_counts.get(0, 0)),
            "nan_count": int(nan_inf["nan_count"]),
            "inf_count": int(nan_inf["inf_count"]),
            "pca_fit_split": "train" if feature_cfg["use_pca"] else "",
            "pca_explained_variance_target": PCA_EXPLAINED_VARIANCE if feature_cfg["use_pca"] else np.nan,
            "pca_n_components_actual": pca_summary.get("pca_n_components_actual", np.nan),
            "pca_explained_variance_actual": pca_summary.get("pca_explained_variance_actual", np.nan),
            "pca_artifact_path": pca_summary.get("pca_artifact_path", ""),
            "status": "OK" if nan_inf["nan_count"] == 0 and nan_inf["inf_count"] == 0 else "WARNING_NAN_INF",
            "action": action,
            "file_size_mb": round(output_path.stat().st_size / (1024 * 1024), 4) if output_path.exists() else np.nan,
            "created_at": timestamp_now(),
            "notes": "HOG32 removed from Stage 03 feature grid.",
        }

        feature_manifest_rows.append(row_summary)
        feature_generation_status_rows.append(row_summary)
        nan_inf_rows.append({
            "dataset_name": DATASET_NAME,
            "patch_set": patch_set,
            "feature_set": feature_set,
            "nan_count": int(nan_inf["nan_count"]),
            "inf_count": int(nan_inf["inf_count"]),
            "status": row_summary["status"],
        })

    del metadata_df, metadata_subset_df, hog_matrices, hsv_matrix, lbp_matrix, pca_matrices
    image_cache.clear()
    gc.collect()

feature_manifest_df = pd.DataFrame(feature_manifest_rows)
feature_generation_status_df = pd.DataFrame(feature_generation_status_rows)
pca_artifact_summary_df = pd.DataFrame(pca_artifact_summary_rows)
crop_read_error_audit_df = pd.DataFrame(
    crop_error_rows,
    columns=["dataset_name", "patch_set", "row_index", "patch_id", "source_image_path", "error"],
)
feature_nan_inf_check_df = pd.DataFrame(nan_inf_rows)

save_dataframe_csv(feature_manifest_df, FEATURE_MANIFEST_PATH, overwrite=OVERWRITE_FEATURES, note="dataset feature manifest")
save_dataframe_csv(feature_generation_status_df, TABLES_DIR / "feature_generation_status.csv", overwrite=OVERWRITE_TABLES)
save_dataframe_csv(pca_artifact_summary_df, TABLES_DIR / "pca_artifact_summary.csv", overwrite=OVERWRITE_TABLES)
save_dataframe_csv(crop_read_error_audit_df, TABLES_DIR / "crop_read_error_audit.csv", overwrite=OVERWRITE_TABLES)
save_dataframe_csv(feature_nan_inf_check_df, TABLES_DIR / "feature_nan_inf_check.csv", overwrite=OVERWRITE_TABLES)

log_dataframe("Feature Manifest", feature_manifest_df)
log_dataframe("PCA Artifact Summary", pca_artifact_summary_df)
log_dataframe("Feature NaN / Inf Check", feature_nan_inf_check_df)
log_dataframe("Crop Read Error Audit", crop_read_error_audit_df)


## 12. Doğrulama ve Özet Tablolar

Üretilen feature dosyalarının split, label, boyut ve NaN/Inf kontrolleri yapılır.


In [ ]:
log_section("08 VALIDATION AND SUMMARY TABLES")

split_count_summary_df = (
    feature_manifest_df
    .groupby(["dataset_name", "patch_set"], as_index=False)
    [["train_row_count", "valid_row_count", "test_row_count"]]
    .first()
)

label_count_summary_df = (
    feature_manifest_df
    .groupby(["dataset_name", "patch_set"], as_index=False)
    [["positive_count", "negative_count"]]
    .first()
)

feature_dimension_summary_df = (
    feature_manifest_df
    .groupby(["dataset_name", "patch_set", "feature_set", "is_pca"], as_index=False)
    [["feature_column_count", "total_column_count", "file_size_mb"]]
    .first()
)

validation_checks_df = pd.DataFrame([{
    "dataset_name": DATASET_NAME,
    "expected_patch_set_count": len(PATCH_SETS),
    "expected_feature_set_count": len(FEATURE_SETS),
    "expected_feature_file_count": len(PATCH_SETS) * len(FEATURE_SETS),
    "actual_feature_file_count": int((feature_manifest_df["status"] == "OK").sum()),
    "nan_problem_file_count": int((feature_manifest_df["nan_count"] > 0).sum()),
    "inf_problem_file_count": int((feature_manifest_df["inf_count"] > 0).sum()),
    "crop_error_count": len(crop_read_error_audit_df),
    "status": (
        "READY_FOR_NEXT_STAGE"
        if (
            len(crop_read_error_audit_df) == 0
            and int((feature_manifest_df["nan_count"] > 0).sum()) == 0
            and int((feature_manifest_df["inf_count"] > 0).sum()) == 0
        )
        else "COMPLETED_WITH_WARNINGS"
    ),
}])

save_dataframe_csv(split_count_summary_df, TABLES_DIR / "split_count_summary.csv", overwrite=OVERWRITE_TABLES)
save_dataframe_csv(label_count_summary_df, TABLES_DIR / "label_count_summary.csv", overwrite=OVERWRITE_TABLES)
save_dataframe_csv(feature_dimension_summary_df, TABLES_DIR / "feature_dimension_summary.csv", overwrite=OVERWRITE_TABLES)
save_dataframe_csv(validation_checks_df, TABLES_DIR / "validation_checks.csv", overwrite=OVERWRITE_TABLES)

log_dataframe("Split Count Summary", split_count_summary_df)
log_dataframe("Label Count Summary", label_count_summary_df)
log_dataframe("Feature Dimension Summary", feature_dimension_summary_df)
log_dataframe("Validation Checks", validation_checks_df)


## 13. Sanity-Check Görselleri

Feature boyutları, satır sayıları, PCA bileşenleri ve örnek patch crop görselleri üretilir.


In [ ]:
log_section("09 SANITY-CHECK FEATURE FIGURES")

# Figure 1 — feature dimensions by feature set
plt.figure(figsize=(12, 6))
plot_df = (
    feature_dimension_summary_df
    .groupby("feature_set", as_index=False)["feature_column_count"]
    .max()
    .sort_values("feature_column_count", ascending=False)
)
plt.bar(plot_df["feature_set"], plot_df["feature_column_count"])
plt.xticks(rotation=75, ha="right")
plt.ylabel("Feature column count")
plt.title(f"{DATASET_DISPLAY_NAME} — Feature Dimension by Feature Set")
plt.tight_layout()
feature_dim_fig_path = FIGURES_DIR / "feature_dimension_by_feature_set.png"
save_current_figure(
    feature_dim_fig_path,
    "Feature Dimension by Feature Set",
    "Maximum feature dimension per feature set.",
)

# Figure 2 — row count by patch set
plt.figure(figsize=(10, 5))
row_df = metadata_input_inventory_df.sort_values("patch_set")
plt.bar(row_df["patch_set"], row_df["row_count"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Rows")
plt.title(f"{DATASET_DISPLAY_NAME} — Row Count by Patch Set")
plt.tight_layout()
row_count_fig_path = FIGURES_DIR / "row_count_by_patch_set.png"
save_current_figure(
    row_count_fig_path,
    "Row Count by Patch Set",
    "Patch metadata row count per patch set.",
)

# Figure 3 — PCA components
if not pca_artifact_summary_df.empty:
    plt.figure(figsize=(10, 5))
    pca_plot_df = pca_artifact_summary_df.copy()
    pca_plot_df["label"] = pca_plot_df["patch_set"] + " / " + pca_plot_df["hog_variant"]
    plt.bar(pca_plot_df["label"], pca_plot_df["pca_n_components_actual"])
    plt.xticks(rotation=75, ha="right")
    plt.ylabel("PCA components")
    plt.title(f"{DATASET_DISPLAY_NAME} — PCA Components by Patch Set")
    plt.tight_layout()
    pca_fig_path = FIGURES_DIR / "pca_components_by_patch_set.png"
    save_current_figure(
        pca_fig_path,
        "PCA Components by Patch Set",
        "Actual PCA component count after 0.95 explained variance threshold.",
    )

# Figure 4 — sample patch sanity grid
sample_patch_set = PATCH_SETS[0]
sample_metadata_path = METADATA_DIR / f"patch_metadata_{sample_patch_set}.csv"
sample_df = pd.read_csv(sample_metadata_path).groupby("patch_label", group_keys=False).head(4)

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
axes = axes.flatten()

for ax, (_, row) in zip(axes, sample_df.iterrows()):
    patch = crop_patch_from_row(row)
    patch_rgb = cv2.cvtColor(patch, cv2.COLOR_BGR2RGB)
    ax.imshow(patch_rgb)
    ax.set_title(f"{row['split']} | label={row['patch_label']}")
    ax.axis("off")

for ax in axes[len(sample_df):]:
    ax.axis("off")

plt.suptitle(f"{DATASET_DISPLAY_NAME} — Sample Cropped Patches ({sample_patch_set})")
plt.tight_layout()
sample_fig_path = FIGURES_DIR / "sample_patch_feature_sanity_grid.png"
save_current_figure(
    sample_fig_path,
    "Sample Patch Feature Sanity Grid",
    "Small crop grid used to confirm crop reconstruction before/after feature extraction.",
)


## 14. Final Durum

Notebook sonunda feature üretimi ve doğrulama sonucu özetlenir.


In [ ]:
log_section("10 FINAL DURUM")

notebook_finished_at = datetime.now()
elapsed_minutes = (notebook_finished_at - NOTEBOOK_STARTED_AT).total_seconds() / 60

notebook_status_df = pd.DataFrame([{
    "dataset_name": DATASET_NAME,
    "notebook_name": NOTEBOOK_NAME,
    "stage_name": STAGE_NAME,
    "started_at": NOTEBOOK_STARTED_AT.strftime("%Y-%m-%d %H:%M:%S"),
    "finished_at": notebook_finished_at.strftime("%Y-%m-%d %H:%M:%S"),
    "elapsed_minutes": round(elapsed_minutes, 2),
    "feature_set_count": len(FEATURE_SETS),
    "patch_set_count": len(PATCH_SETS),
    "feature_file_count": len(feature_manifest_df),
    "pca_artifact_count": len(pca_artifact_summary_df),
    "crop_error_count": len(crop_read_error_audit_df),
    "status": validation_checks_df["status"].iloc[0],
}])

notebook_status_df = save_dataframe_csv(
    notebook_status_df,
    TABLES_DIR / "notebook_status.csv",
    overwrite=OVERWRITE_TABLES,
)

log_dataframe("Notebook Status", notebook_status_df)
log_output(f"Final status: {notebook_status_df['status'].iloc[0]}")
log_output("Feature engineering notebook tamamlandı.")
